# 05 SHAP-蜂窝图解释（任务书：SHAP、分区主控因子变化）

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import shap

# ---- bootstrap: src ----
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_config, ensure_dir, safe_col
from src.spatial_cv import make_spatial_groups, get_group_kfold
from src.models import fit_oof_with_spatialcv
from src.shap_plots import shap_importance_combo

cfg = load_config(ROOT / "config" / "config.yaml")
OUT = ensure_dir(ROOT / "outputs")
SHAP_DIR = ensure_dir(OUT / "shap")

# ---- data ----
p_model = OUT / "clean_data.parquet"
df = pd.read_parquet(p_model )

bcf_col = safe_col(df, cfg["data"]["columns"]["target_bcf"])
cv = get_group_kfold(cfg)

preferred = [m for m in ["xgb", "rf"] if m in cfg["modeling"]["models"]]
if not preferred:
    raise RuntimeError("请在 config.yaml 的 modeling.models 至少包含 xgb 或 rf")
m = preferred[0]

group_keys = ["crop", "ph_bin"] if "crop" in df.columns else ["ph_bin"]

# 控制速度
sample_rows = int(cfg.get("shap", {}).get("sample_rows", 800))
top_k = int(cfg.get("shap", {}).get("top_k", 15))
max_display = int(cfg.get("shap", {}).get("max_display", 20))

def _safe_key(x):
    s = str(x)
    return s.replace(" ", "").replace("(", "").replace(")", "").replace(",", "_").replace("'", "")

summary_rows = []

for key, sub in df.groupby(group_keys, dropna=False):
    sub = sub.reset_index(drop=True)
    gsub = make_spatial_groups(sub, cfg)
    res = fit_oof_with_spatialcv(sub, cfg, gsub, m, cv)

    pipe = res.best_estimator  # sklearn Pipeline
    if not hasattr(pipe, "named_steps") or "pre" not in pipe.named_steps:
        raise RuntimeError("best_estimator 不是包含 'pre' 的 Pipeline（请确认你训练代码的 pipeline 命名为 pre）")

    # ---- 原始特征（给 pre 用，允许字符串列存在）----
    X_all = sub.drop(columns=[bcf_col], errors="ignore").reset_index(drop=True)
    if len(X_all) > sample_rows:
        X_used = X_all.sample(sample_rows, random_state=int(cfg["project"]["seed"])).reset_index(drop=True)
    else:
        X_used = X_all

    pre = pipe.named_steps["pre"]

    # ---- 关键：转成纯数值矩阵（不会再出现 str-str 相减）----
    Xt = pre.transform(X_used)

    # feature names（展开后的 one-hot 等）
    try:
        feat_names = list(pre.get_feature_names_out())
    except Exception:
        # 兜底：没有就用 f0..fp
        feat_names = [f"f{i}" for i in range(Xt.shape[1])]

    # 取最终模型（pipe 最后一步）
    last_step_name = list(pipe.named_steps.keys())[-1]
    est = pipe.named_steps[last_step_name]

    # ---- TreeExplainer：对 xgb/rf 稳定 ----
    explainer = shap.TreeExplainer(est)
    sv = explainer.shap_values(Xt)

    # 兼容有些版本返回 list
    if isinstance(sv, list):
        sv = sv[0]

    sv = np.asarray(sv, dtype=float)
    # 强制对齐
    if sv.shape[0] != Xt.shape[0]:
        sv = sv[: Xt.shape[0], :]
    if sv.shape[1] != Xt.shape[1]:
        sv = sv[:, : Xt.shape[1]]

    key_str = "_".join(_safe_key(k) for k in (key if isinstance(key, tuple) else (key,)))
    out_png = SHAP_DIR / f"shap_combo_{key_str}_{m}.png"

    shap_importance_combo(
        shap_values=sv,
        X=Xt,  # 注意：这里已经是纯数值矩阵
        feature_names=feat_names,
        out_png=out_png,
        title=f"{key} | {m} | n={sv.shape[0]}",
        max_display=max_display,
    )
    print("saved:", out_png)

    mean_abs = np.mean(np.abs(sv), axis=0)
    idx = np.argsort(mean_abs)[::-1][:top_k]
    for i in idx:
        summary_rows.append({
            "group": str(key),
            "model_for_shap": m,
            "feature": feat_names[i] if i < len(feat_names) else f"f{i}",
            "mean_abs_shap": float(mean_abs[i]),
        })

rank = pd.DataFrame(summary_rows)
rank.to_excel(OUT / "shap_rank_compare.xlsx", index=False)
rank.head(20)


saved: C:\My File\cao\soil\soil_task_project\outputs\shap\shap_combo_corn_acid_xgb.png
saved: C:\My File\cao\soil\soil_task_project\outputs\shap\shap_combo_corn_alkaline_xgb.png
saved: C:\My File\cao\soil\soil_task_project\outputs\shap\shap_combo_corn_neutral_xgb.png
saved: C:\My File\cao\soil\soil_task_project\outputs\shap\shap_combo_corn_strong_acid_xgb.png
saved: C:\My File\cao\soil\soil_task_project\outputs\shap\shap_combo_potato_acid_xgb.png
saved: C:\My File\cao\soil\soil_task_project\outputs\shap\shap_combo_potato_alkaline_xgb.png
saved: C:\My File\cao\soil\soil_task_project\outputs\shap\shap_combo_potato_neutral_xgb.png
saved: C:\My File\cao\soil\soil_task_project\outputs\shap\shap_combo_potato_strong_acid_xgb.png


,group,model_for_shap,feature,mean_abs_shap
0,"('corn', 'acid')",xgb,Distance from pollution source,0.002809
1,"('corn', 'acid')",xgb,pH,0.001277
2,"('corn', 'acid')",xgb,Particle content,0.001070
3,"('corn', 'acid')",xgb,P,0.001020
4,"('corn', 'acid')",xgb,water content,0.000880
5,"('corn', 'acid')",xgb,clay content,0.000569
6,"('corn', 'acid')",xgb,K,0.000550
7,"('corn', 'acid')",xgb,CEC,0.000547
8,"('corn', 'acid')",xgb,bulk density,0.000500
9,"('corn', 'acid')",xgb,Mn,0.000433
